In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpad\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestriction\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeTime\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc3\userspace\miao\cluster"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc3\userspace\miao\cluster"


In [4]:
static var myBatch = ExecutionQueues[2];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE-LikeEL-3D");

In [5]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE-LikeEL-3D");

Project name is set to 'Droplet-EE-LikeEL-3D'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

## Create grid

In [7]:
public static class GridFactory {
 
    public static Grid3D GenerateGrid() { 
        double xSize = 1;
        double yTop = 1 - 20.0 / 176.7;
        double yBottom = -20.0 / 176.7;
        int kelem = 8;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 2 + 1);
        double[] Znodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
                var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");
                grd.EdgeTagNames.Add(5, "FreeSlip_front");
                grd.EdgeTagNames.Add(6, "FreeSlip_back");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;
                    if(Math.Abs(X[2] - xSize) <= 1.0e-8)
                        et = 5;
                    if(Math.Abs(X[2] + xSize) <= 1.0e-8)
                        et = 6;

                    return et;
                });

                return grd;
     }
 
 }

In [8]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double pJump(double[] X) {");
           stw.WriteLine("    return 0.046 / 0.4;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return (((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() + (X[2] - 0.0).Pow2()) - 0.16);");
           //stw.WriteLine("    return -(X[0] - 0);");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] + 0);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_pJump(){
        return new Formula("BoundaryValues.pJump", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [9]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;
            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            int D = 3;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            //C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE-LikeEL-3D";
            C.ProjectName = "Droplet";
            C.SessionName = "Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS";
            //C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            if(D == 3) {
                C.FieldOptions.Add("DisplacementZ", new FieldOpts() {
                    Degree = p,
                    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
                });
            }

            // Physical Parameters
            // ===================
            #region physics

            double scale = 4.4175e-4; // For a droplet with radius r = 176.7 micrometre
            C.PhysicalParameters.rho_A = 10 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false; //??

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale, // Real Viscosity
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_pJump());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_Zero());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");
            C.AddBoundaryValue("FreeSlip_front");
            C.AddBoundaryValue("FreeSlip_back");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = true;
            //C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 2});
            //C.AMR_startUpSweeps = 2;
            C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = 2, levelSets = new[] { 0, 1 }, FieldWidth = 1 });
            C.AMR_startUpSweeps = 2;

            #endregion

            C.DynamicLoadBalancing_On = true;
            C.DynamicLoadBalancing_Period = 10;
            C.DynamicLoadBalancing_RedistributeAtStartup = true;

            C.GridPartType = GridPartType.METIS;

            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            //double dt = 5e-3;
            double dt = 2e-4;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 10000;
            C.saveperiod = 1;

            #endregion

            return C;
        }

## Send and run jobs

In [10]:
    var C_CTRLFILE = Aland(2, 0);//Create control file.

Grid Edge Tags changed.


In [11]:
    var JobLocal = C_CTRLFILE.CreateJob();

In [12]:
JobLocal.Status

PreActivation

In [13]:
    JobLocal.NumberOfMPIProcs = 16;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate();
    JobLocal.ShowOutput();
    //BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS ... 
Opening existing database 'C:\Users\miao\Documents\Database\Droplet-EE-LikeEL-3D'.
Set Database: { Session Count = 42; Grid Count = 250; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D }
Grid is not in database yet...
Grid successfully saved: 799b50ca-20ed-4ec6-8873-b3b3d479d075


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-3D-ZwoLevelSetSolver2024Feb12_161000.240589
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [16]:
JobLocal.ShowOutput();

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [12]:
databases[0].Sessions

#0: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS*	2/12/2024 4:12:48 PM	f36a3a9c...
#1: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS*	2/10/2024 5:54:07 PM	da5d3b07...
#2: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS*	2/10/2024 10:30:02 AM	7bfd5d0d...
#3: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS*	2/10/2024 9:59:44 AM	09eb6a7c...
#4: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS*	2/9/2024 1:33:54 PM	e6c3349f...
#5: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS*	1/11/2024 3:16:27 PM	3ccdeb6a...
#6: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-6x6x3AMR3-Print-RemoveRestriction-LoadBalancing-METIS*	12/23/2023 4:05:32 PM	4eccd2ce...
#7: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-6x6x3AMR3-Print-RemoveRestriction-LoadBalancing-METIS*	12/22/20

In [13]:
wmg.Sessions[0].Timesteps.Count

5

In [14]:
var outPath = wmg.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-3D__Droplet-3D-Demo-8x8x4AMR2-Print-WithRestriction-LoadBalancing-METIS__f36a3a9c-5029-4288-862e-0f429dad73f7


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

## Restart

In [35]:
databases[0].Sessions

#0: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS_Restart*	1/13/2024 2:54:06 PM	eea7b8a4...
#1: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS*	1/12/2024 4:08:11 PM	db5e3440...
#2: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-ParMETIS*	1/12/2024 4:03:20 PM	061ac9a0...
#3: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-Hilbert*	1/12/2024 3:59:10 PM	2b8eaa24...
#4: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-Hilbert*	1/11/2024 3:22:58 PM	719fd65b...
#5: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-clusterHilbert*	1/11/2024 3:21:32 PM	2a170e04...
#6: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS*	1/10/2024 5:34:02 PM	156a2aae...
#7: Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestrictio

In [36]:
var Sloc = databases[0].Sessions[0];
Sloc

Droplet-EE-LikeEL-3D	Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS_Restart*	1/13/2024 2:54:06 PM	eea7b8a4...

In [37]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [38]:
c2.GetType()

ZwoLevelSetSolver.ZLS_Control

In [39]:
c2.GridGuid

65ffae1f-7482-4389-9ce6-d7d0dd121cb6

In [40]:
Sloc.Timesteps.Last().GridID

f9c89d2a-b6a2-4310-9f05-58ed304b9132

In [42]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [43]:
c2.GridGuid

f9c89d2a-b6a2-4310-9f05-58ed304b9132

In [45]:
c2.AMR_startUpSweeps = 0;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;

In [46]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

PreActivation

In [47]:
JobLocal2.NumberOfMPIProcs = 8;
JobLocal2.Activate();
JobLocal2.ShowOutput();

Deploying job Droplet-3D-Demo-8x8x4AMR2-Print-RemoveRestriction-LoadBalancing-METIS_Restart_Restart ... 
Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\Droplet-EE-LikeEL-3D-ZwoLevelSetSolver2024Jan14_123302.492291
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.

Starting external console ...
(You may close the new window at any time, the job will continue.)
